In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3

In [2]:
df = pd.read_csv("dataset/football_matches_dataset.csv")

In [3]:
clean_df = df.drop(columns=["Unnamed: 0"])

In [4]:
clean_df["datetime"] = pd.to_datetime(clean_df["datetime"])

In [5]:
conditions = [
    clean_df["home_score"] > clean_df["away_score"],
    clean_df["home_score"] < clean_df["away_score"]
]

choices = [
    "Home Win",
    "Away Win"
]

clean_df["match_result"] = np.select(
    conditions,
    choices,
    default="Draw"
)

#this step it could be done with a for loop (was my firts test actually), but choosing np.select() will improve speed and let me apply pandas.   BORRAR
# np.select(condlist, choicelist, default=...) variables as condition and chioces, are not reserved words for this np function, we could use a,b. 
# BUT Without specifying default, np.select() uses 0 as the default value. 

In [6]:
clean_df["goal_difference"] = clean_df["home_score"] - clean_df["away_score"]

### 3.7 recent_form_difference
sums up scores (goals) of last_4_matches

* Relevant variables:

- (not 15-16 - total number of points home/away team have gained in their previous 4 matches)
- 19-20 - total number of goals home/away team have scored in their previous 4 matches 
- 23-24 - total number of goals home/away team have conceded (goals against) in their previous 4 matches

(each) - can be NaN in case of the first matches of the season when teams have not played enough matches this season yet

In [7]:
# Speaking of Points - Form could be calculated as the sum of the last 4 matches points (a+h) 
#clean_df.home_points_last_4_matches
#clean_df.away_points_last_4_matches
newcol = clean_df.home_points_last_4_matches + clean_df.away_points_last_4_matches
#clean_df["recent_form_difference"] = newcol

# Speaking of Scores - Form could be calculated as the sum of the last 4 matches scores: (a+h) 
# the result per game is goals_L4m - goals_againstL4m; sum(a+h)
#clean_df.away_goals_last_4_matches
#clean_df.away_goals_against_last_4_matches
#clean_df.home_goals_last_4_matches
#clean_df.home_goals_against_last_4_matches
newcola = clean_df.away_goals_last_4_matches - clean_df.away_goals_against_last_4_matches
newcolh = clean_df.home_goals_last_4_matches - clean_df.home_goals_against_last_4_matches

newcols = newcolh + newcola
#clean_df["recent_form_difference"] = newcols

print(newcols.tail(10))


16322   -7.0
16323   -5.0
16324   -2.0
16325    3.0
16326   -4.0
16327    3.0
16328   -2.0
16329   -7.0
16330   -2.0
16331    9.0
dtype: float64


### Methodological Note

The original implementation explored two different ways of measuring recent form:

- Using the total points from the last 4 matches.
- Using the goal balance from the last 4 matches.

However, neither approach was completed as the final feature, and both calculations combined the values from the home and away teams instead of comparing them.

Since our Research Question is:

> **Does recent team form influence match outcomes?**

we need to compare the recent form of both teams before the match.

For this reason, we calculate:

```python
clean_df["recent_form_difference"] = (
    clean_df["home_points_last_4_matches"]
    - clean_df["away_points_last_4_matches"]
)
```

This feature is easier to interpret:

- Positive values → Home team was in better recent form.
- Negative values → Away team was in better recent form.
- Values close to zero → Both teams had similar recent form.

This implementation better matches the objective of Research Question 2.